# Chapter 15 — Speeding Up All the Things with Binary Search Trees

**Source:** Wengrow, *A Common-Sense Guide to Data Structures and Algorithms*, Ch. 15

## The problem this chapter solves
We want data that **stays in order** *and* has fast search, insert and delete.

| Structure | Keeps order? | Search | Insert | Delete |
|---|---|---|---|---|
| Ordered array | ✅ | O(log N) (binary search) | **O(N)** (shifting) | **O(N)** (shifting) |
| Hash table (dict) | ❌ | O(1) | O(1) | O(1) |
| **Binary search tree** | ✅ | **O(log N)** | **O(log N)** | **O(log N)** |

The BST gives you order *plus* fast changes. Its advantage over an ordered array is **insertion and deletion**.

## 1. Trees: vocabulary

A **tree** is node-based like a linked list, but a node can link to **several** nodes.

- **Root**: the top node (trees are drawn upside down).
- **Parent / child**: a node and the nodes directly below it.
- **Ancestors / descendants**: all nodes above / below a node.
- **Level**: a row of the tree.
- **Balanced**: at every node, the left and right subtrees have (about) the same number of nodes. If one side has more, the tree is **imbalanced**.

## 2. Binary search tree (BST)

- **Binary tree:** every node has 0, 1 or 2 children.
- **Binary *search* tree:** a binary tree where also
  - each node has **at most one left** and **one right** child, and
  - **all left descendants < node < all right descendants**. This holds at *every* node, not just for the direct children.

Example BST used throughout the chapter:

            50
         /      \
       25        75
      /  \      /  \
    10    33   56    89
    / \   / \  / \   / \
   4  11 30 40 52 61 82 95

In [ ]:
class TreeNode:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left_child = left
        self.right_child = right


# Small manual example
root = TreeNode(50, TreeNode(25), TreeNode(75))
print(root.value, root.left_child.value, root.right_child.value)

## 3. Search: O(log N)

Algorithm (start at the root):
1. If this node is the value → found.
2. If the value is **smaller** → search the **left** subtree.
3. If it's **bigger** → search the **right** subtree.
4. If you reach an empty spot (`None`) → the value isn't in the tree.

Each step **throws away half** of the remaining tree → **O(log N)**. This is the same halving idea as binary search and your chessboard exercise.

**Another way to see it: log N levels.**
Each full level **doubles** the number of nodes (1, 2, 4, 8, 16 …). So a balanced tree of N nodes has about **log₂ N levels**, and a search moves down one level per step.
Example: 4 full levels hold 15 nodes; 5 full levels hold 31 nodes (log₂ 31 ≈ 5).

Recursion fits naturally because a tree can be arbitrarily deep (Chapter 10), and every subtree is itself a BST.
The `node is None` base case covers "the value isn't here": we tried to go into a child that doesn't exist.

In [ ]:
def search(search_value, node):
    # Base cases: fell off the tree (not found) OR found it
    if node is None or node.value == search_value:
        return node
    elif search_value < node.value:
        return search(search_value, node.left_child)
    else:
        return search(search_value, node.right_child)

## 4. Insertion: O(log N), the big win over ordered arrays

Search down as if looking for the value. When you reach a node with **no child on the side you need**, attach the new node there.
Cost: a search (log N) **+ 1 step** → **O(log N)**. An ordered array needs O(N) because of shifting.

Example: insert 45 → 50 (go left) → 25 (go right) → 33 (go right) → 40 (no right child) → becomes 40's right child.

### The order of insertion matters!
- Random order → the tree is usually well balanced → O(log N).
- **Sorted order** (1, 2, 3, 4, 5) → every node goes right → the tree is basically a **linked list** → search becomes **O(N)**.
- Same values inserted as 3, 2, 4, 1, 5 → balanced.

So the worst case for a BST is O(N), and the best/typical case is O(log N). (It's the same lesson as Quicksort: sorted input is the bad case.)
To turn a sorted array into a BST, **shuffle it first**.

In [ ]:
def insert(value, node):
    if value < node.value:
        if node.left_child is None:
            node.left_child = TreeNode(value)        # base case: free spot found
        else:
            insert(value, node.left_child)
    elif value > node.value:
        if node.right_child is None:
            node.right_child = TreeNode(value)
        else:
            insert(value, node.right_child)
    # (equal values are ignored: a BST holds unique values)


def build_tree(values):
    """Insert values one by one, in the given order. Returns the root."""
    root = TreeNode(values[0])
    for v in values[1:]:
        insert(v, root)
    return root


def show(node):
    """Print the tree top-down. L = left child, R = right child."""
    def walk(n, indent, label):
        if n is None:
            return
        print(" " * indent + label + str(n.value))
        walk(n.left_child, indent + 4, "L: ")
        walk(n.right_child, indent + 4, "R: ")
    walk(node, 0, "")


root = build_tree([50, 25, 75, 10, 33, 56, 89, 4, 11, 30, 40, 52, 61, 82, 95])
show(root)

print(search(61, root).value)    # 61
print(search(60, root))          # None

# Sorted insertion order -> a "linked list" shaped tree:
show(build_tree([1, 2, 3, 4, 5]))

## 5. Deletion: O(log N), the hardest part of the chapter

Find the node, then use one of these rules:

| Node to delete has… | What to do |
|---|---|
| **no children** | just remove it |
| **one child** | remove it and **plug the child into its place** |
| **two children** | replace it with its **successor node** |

**Successor** = the smallest value that is *bigger* than the deleted value (the next number in sorted order).
**How to find it:** go to the **right child** once, then keep going **left** until there's no more left child.

**Extra rule:** if the successor has a **right child**, that child becomes the **left child of the successor's old parent** (so it isn't left dangling).

### How the code does this (the clever part)
- `delete` **returns the (new) root of the subtree** it was called on. The parent sets `node.left_child = delete(...)` or `node.right_child = delete(...)`. Most of the time this returns the same child, so nothing changes. Only at the deleted node does it return something different (its child, or `None`).
- For two children, the code doesn't move node objects. `lift` **copies the successor's value** into the node being deleted, then removes the original successor node by returning its right child to the successor's parent.

The book says itself: "the delete function is one of the most complex pieces of code in the book." Trace it on paper with the example tree.

In [ ]:
def delete(value_to_delete, node):
    if node is None:                                   # value not found
        return None

    elif value_to_delete < node.value:
        node.left_child = delete(value_to_delete, node.left_child)
        return node                                    # this node stays

    elif value_to_delete > node.value:
        node.right_child = delete(value_to_delete, node.right_child)
        return node

    else:  # this is the node to delete
        if node.left_child is None:                    # 0 children or only a right child
            return node.right_child
        elif node.right_child is None:                 # only a left child
            return node.left_child
        else:                                          # 2 children -> successor
            node.right_child = lift(node.right_child, node)
            return node


def lift(node, node_to_delete):
    if node.left_child:                                # keep going left to find successor
        node.left_child = lift(node.left_child, node_to_delete)
        return node
    else:                                              # this node IS the successor
        node_to_delete.value = node.value              # copy its value up
        return node.right_child                        # its right child takes its place


root = build_tree([50, 25, 75, 10, 33, 56, 89, 4, 11, 30, 40, 52, 61, 82, 95])
root = delete(4, root)     # no children
root = delete(10, root)    # one child (11 moves up)
root = delete(56, root)    # two children (successor 61)
root = delete(50, root)    # root! successor is 52
show(root)                 # note: always reassign root = delete(...)

## 6. Traversal: visiting every node, O(N)

**Inorder traversal** = left subtree → **visit node** → right subtree.
On a BST this visits the values in **sorted order**, which is how the book-title app prints titles alphabetically.

It runs through the call stack exactly as in Chapter 10: each call "waits" on the stack while its left subtree is processed.
Traversal visits all N nodes → **O(N)**.

Strings compare alphabetically in Python (`"Alice" < "Moby"` is `True`), so the same BST code works for book titles.

(Two other orders, **preorder** and **postorder**, come up in the exercises.)

## 7. Summary

| Operation | Ordered array | BST (balanced) |
|---|---|---|
| Search | O(log N) | O(log N) |
| Insert | O(N) | **O(log N)** |
| Delete | O(N) | **O(log N)** |
| Print in order | O(N) | O(N) (inorder traversal) |

- BSTs are worth it when data is **ordered *and* changes often**.
- Worst case (imbalanced, e.g. sorted inserts) degrades to O(N).
- Python has no built-in BST. `dict`/`set` are hash tables (no order by value), and `sorted()` + `bisect` give you an ordered array. Self-balancing trees exist in libraries (e.g. `sortedcontainers`). Here you're learning the mechanism.

In [ ]:
def traverse_and_print(node):          # INORDER
    if node is None:
        return
    traverse_and_print(node.left_child)
    print(node.value)
    traverse_and_print(node.right_child)


# The book-title tree (inserting in this order produces the tree from the book):
#
#                  "Moby Dick"
#               /              \
#    "Great Expectations"   "Robinson Crusoe"
#        /         \            /          \
# "Alice in   "Lord of    "Pride and    "The Odyssey"
#  Wonderland" the Flies"  Prejudice"
books = build_tree([
    "Moby Dick",
    "Great Expectations", "Robinson Crusoe",
    "Alice in Wonderland", "Lord of the Flies", "Pride and Prejudice", "The Odyssey",
])
traverse_and_print(books)              # alphabetical!

## Exercises (book solutions: p. 456)

### Exercise 1: draw a BST (pen and paper)
**Input:** start with an **empty** tree and insert these numbers **in this exact order**: `[1, 5, 9, 2, 4, 10, 6, 3, 8]`.
The first number (1) becomes the root, and each next number follows the insertion rule (smaller → left, bigger → right).
**Task:** draw the resulting tree. Then check yourself with `show(build_tree([...]))`.

### Exercise 2: reasoning, no code
**Input:** a **well-balanced** BST holding **1,000** values.
**Question:** what's the **maximum** number of steps to search for a value? (Think: how many levels does it have?)

### Exercise 3: greatest value
**Input:** the root of a BST, e.g. the 50-tree from above.
**Task:** write `greatest(node)` that returns the largest value. Use the BST rule; you don't need to look at every node.

### Exercise 4: preorder traversal (predict, then run)
**Input:** the book-title tree (`books`).
**Code:** preorder = **visit node first**, then left, then right.
**Task:** write down the order in which titles are printed *before* running it.

### Exercise 5: postorder traversal (predict, then run)
**Input:** the same book-title tree.
**Code:** postorder = left, right, then **visit node last**.
**Task:** write down the printed order first, then run it.

In [ ]:
# Exercise 1 — draw first, then check:
# show(build_tree([1, 5, 9, 2, 4, 10, 6, 3, 8]))

# Exercise 2 — write your answer here as a comment:
# max steps for 1000 balanced values = ?

# Exercise 3
def greatest(node):
    pass

root = build_tree([50, 25, 75, 10, 33, 56, 89, 4, 11, 30, 40, 52, 61, 82, 95])
# print(greatest(root))      # expect 95

In [ ]:
# Exercise 4 — PREORDER (write your predicted order first!)
def preorder_print(node):
    if node is None:
        return
    print(node.value)
    preorder_print(node.left_child)
    preorder_print(node.right_child)

# preorder_print(books)


# Exercise 5 — POSTORDER (write your predicted order first!)
def postorder_print(node):
    if node is None:
        return
    postorder_print(node.left_child)
    postorder_print(node.right_child)
    print(node.value)

# postorder_print(books)